# edacopilot demo: the Section 7.4 worked example

This notebook walks `ARCHITECTURE.md` Section 7.4's worked example end to end through the Python API — the same API the ipywidgets panel drives, so everything below is exactly what pressing the panel's buttons would do.

It covers, in order:
1. A seeded dataset that reproduces Section 7.4's divergence (two independent groups, right-skewed, personas disagree).
2. An override with a typed reason (Section 7.5's friction on a `CAVEAT` candidate).
3. A post-hoc after a significant omnibus (Section 12.3), on a second, three-group dataset.
4. An undo.

Everything runs in **deterministic mode** — no LLM call anywhere (Rule 9). M9 is the milestone that adds one.

In [ ]:
import numpy as np
import pandas as pd

import edacopilot as eda

## Settings

**Deterministic by default.** This notebook runs with no LLM call anywhere (Rule 9), which is also why the free-text `session.ask("run post-hoc comparisons")` cell below is safe to run without an API key. To see the same notebook with a live LLM's rationales instead of the deterministic templates, set `DETERMINISTIC_MODE = False` below and make sure `GEMINI_API_KEY` (or your configured provider's key) is set in the environment -- but the version this repository tests in CI always runs with it `True`.

In [ ]:
DETERMINISTIC_MODE = True  # set False to use a live LLM instead (needs an API key)
LLM_CONFIG = {"llm": {"deterministic_mode": DETERMINISTIC_MODE}}

## 1. The worked example: income by gender

Same construction as `tests/scenarios/generators.py::worked_example_7_4` (seed chosen so every check lands on the status Section 7.4's table gives): two groups, n = 38 and 41, strong right skew.

In [ ]:
rng = np.random.default_rng(121)
income_df = pd.DataFrame(
    {
        "income": np.concatenate([rng.lognormal(10.5, 0.45, 38), rng.lognormal(10.6, 0.62, 41)]),
        "gender": ["F"] * 38 + ["M"] * 41,
    }
)
income_df.describe()

In [ ]:
session = eda.start(income_df, name="m8_demo", display=False, config=LLM_CONFIG)
proposal = session.ask(
    goal="compare_groups",
    outcome="income",
    group="gender",
    design="independent",
    confirmed_by_user=["design"],
)
print(proposal.to_text())

The personas disagree, and the card says so rather than picking for you (Section 8.3). Professor and Consultant land on `mann_whitney`; the Maverick proposes `yuen_trimmed_t` instead, for a robust mean-based reading of the same data.

## 2. An override, with a typed reason

Suppose the analysis has to match a legacy report that used Student's t-test. `student_t` is a `CAVEAT` candidate here (normality FAIL, equal variance FAIL) that no persona picked, so Section 7.5's friction applies: choosing it needs a typed reason, which is stored in the provenance log and written into the exported code as a comment.

In [ ]:
override_card = session.override(
    "student_t",
    reason="Comparing against a legacy report that used the Student t-test.",
)
print(override_card.to_text())

## 3. Undo

M11 (outliers, transforms, quality) is not built yet, so there is no cleaning-stage button to press — but the mechanism `undo` moves is already there. `session.record_step(stage="quality", new_data=...)` stands in for the cleaning step M11 will add, exactly as the `undo_then_branch` golden transcript does, so this notebook demonstrates undo without implying a stage that does not exist yet.

In [ ]:
trimmed = session.data.query("income < 120_000")
step = session.record_step(stage="quality", new_data=trimmed, label="trimmed the top tail")
print(
    f"Step {step.step_id} created version {step.output_version} from "
    f"{step.input_version}: {len(session.data)} rows, "
    f"down from {len(session.original())}."
)

In [ ]:
undo_card = session.undo()
print(undo_card.to_text())
assert session.version == "v0"

## 4. A post-hoc after a significant omnibus

Section 7.4's dataset only has two groups, so there is no omnibus to follow up. A second, three-group dataset (three clinics, heavy right skew, a real difference in wait times) shows the other half of Section 12.3: a significant omnibus offers its paired post-hoc as a button, which is a proposal like any other — it still waits for acceptance.

In [ ]:
rng = np.random.default_rng(4)
clinics_df = pd.DataFrame(
    {
        "wait_minutes": np.concatenate(
            [rng.exponential(2, 40), rng.exponential(3, 40), rng.exponential(6, 40)]
        ),
        "clinic": ["A"] * 40 + ["B"] * 40 + ["C"] * 40,
    }
)
clinic_session = eda.start(clinics_df, name="m8_demo_clinics", display=False, config=LLM_CONFIG)
clinic_proposal = clinic_session.ask(
    goal="compare_groups",
    outcome="wait_minutes",
    group="clinic",
    design="independent",
    confirmed_by_user=["design"],
)
print(clinic_proposal.to_text())

In [ ]:
omnibus_result = clinic_session.accept("professor")
print(omnibus_result.to_text())

The omnibus (`kruskal_wallis`) is significant, so a "Run post-hoc comparisons" button appears among the result's actions. Asking for it is a proposal, not an execution — the pairwise test still waits for `accept()`.

In [ ]:
post_hoc_proposal = clinic_session.ask("run post-hoc comparisons")
print(post_hoc_proposal.to_text())

In [ ]:
post_hoc_result = clinic_session.accept()
print(post_hoc_result.to_text())

## Recap

Nothing above ran until it was accepted (rule 3): the first proposal card lists six candidates and runs none of them; the override executed exactly one thing, with its reason on record; the undo left `v0` untouched and reachable; and the post-hoc waited for its own accept even though the omnibus that licensed it had already run.
